# 01 — Veri Şeması ve Genel Özet

Bu notebook sentetik wetstock veri ambarının **ilk tanışma** adımıdır.

**Kapsam:**
- 8 operasyonel tablonun yüklenmesi
- Satır/kolon/null özeti
- İstasyon ve tank dağılımı
- Tarih aralığı ve granülarite

> `ground_truth/` klasörünü bu aşamada **kullanmıyoruz** — anomalileri kendimiz keşfedeceğiz.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all, summary_table, DATA_DIR
from utils.plots import setup_style

setup_style()
sns.set_theme(style='whitegrid')
print('Veri klasörü:', DATA_DIR)
dfs = load_all()
list(dfs.keys())

In [ ]:
# Tablo özeti
ozet = summary_table(dfs)
display(ozet)

In [ ]:
# Her tablonun kolonları
for name, df in dfs.items():
    print('=' * 60)
    print(f'{name}: {df.shape[0]:,} satır x {df.shape[1]} kolon')
    print('Kolonlar:', list(df.columns))
    print(df.dtypes)
    print()

In [ ]:
# İstasyon × tank matrisi
stations = dfs['stations']
tanks = dfs['tanks']
print(stations.to_string(index=False))
print()
print('Tank sayısı istasyon başına:')
print(tanks.groupby('istasyon_kodu').size().to_string())

In [ ]:
# Ürün ve kapasite dağılımı
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tanks['akaryakit_turu'].value_counts().plot(kind='bar', ax=axes[0], title='Ürün dağılımı')
tanks.boxplot(column='kapasite', by='istasyon_kodu', ax=axes[1])
axes[1].set_title('Kapasite dağılımı (istasyon)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Tarih aralığı
daily = dfs['daily']
tx = dfs['transactions']
print('Daily  :', daily.tarih.min(), '→', daily.tarih.max(), f'({daily.tarih.nunique()} gün)')
print('TX     :', tx.satis_zamani.min(), '→', tx.satis_zamani.max())
print('Beklenen daily satır (32 tank × 90 gün):', 32 * 90)
print('Gerçek daily satır:', len(daily))

In [ ]:
# Granülarite tablosu
gran = pd.DataFrame([
    ['transactions', 'tekil satış', 'değişken (~100+/gün/tank)'],
    ['ue1t_30min', '30 dakika', '48/gün/tank'],
    ['inventory_30min', '30 dakika', '48/gün/tank'],
    ['daily', 'günlük', '1/gün/tank'],
    ['deliveries', 'olay bazlı', 'dolum olduğunda'],
], columns=['tablo', 'granülarite', 'beklenen'])
display(gran)

## Sonuç

- Veri **yıldız şeması**: `istasyon_kodu + tank_no` ile tüm tablolar birbirine bağlanır.
- Sonraki notebook: katman tutarlılığı doğrulaması (`02_katman_tutarliligi.ipynb`).